# 02 — Read Source Blob via SAS Token
**Day 1 | Part 7.4**

Reads pre-loaded source data from the shared external Azure Blob Storage
(`dataenggdailystorage`) using credentials stored in **your** Key Vault.

**This storage is NOT your storage account** — it is a shared read-only source
provided for the project. You have read + list access only via the SAS token.

**Secrets to add in your Key Vault before running this notebook:**

| Secret Name | Value |
|---|---|
| `source-storage-account` | `dataenggdailystorage` |
| `source-container` | `source` |
| `source-sas-token` | Provided during the session |

> **Important:** This notebook uses `wasbs://` protocol — NOT `abfss://`.
> SAS tokens require `wasbs://`. Using `abfss://` will give an auth error.

## Cell 1 — Load source blob credentials from Key Vault

> All 3 values come from Key Vault — nothing is hardcoded.
> Add these secrets before running (Day 1 Part 7.4 in the setup guide):
> - `source-storage-account` → `dataenggdailystorage`
> - `source-container` → `source`
> - `source-sas-token` → provided during the session

In [ ]:
SCOPE = "kv-ev-scope"

# Load all 3 source blob credentials from Key Vault
try:
    STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
    CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
    SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

    print(f"Storage account : {STORAGE_ACCOUNT}")
    print(f"Container       : {CONTAINER}")
    print(f"SAS token       : [REDACTED]")
    print("All source blob credentials loaded from Key Vault — OK")

except Exception as e:
    print(f"ERROR: {e}")
    print("\nSecrets missing from Key Vault. Add these 3 secrets first:")
    print("  Portal → Key vaults → kv-ev-intelligence-dev → Secrets → + Generate/Import")
    print("  source-storage-account  →  dataenggdailystorage")
    print("  source-container        →  source")
    print("  source-sas-token        →  <provided during session>")
    raise

# Configure Spark to use SAS token for the external storage account
spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)
print("Spark SAS config set — OK")

## Cell 2 — List top-level folders in the source container

In [ ]:
print(f"Listing top-level folders in {CONTAINER}/\n")
try:
    base_path = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/"
    items = dbutils.fs.ls(base_path)

    for item in items:
        item_type = "DIR " if item.size == 0 else "FILE"
        print(f"  [{item_type}] {item.name:<50} {item.size:>10} bytes")

    print(f"\n  Total: {len(items)} items found at root level")

except Exception as e:
    print(f"ERROR: {e}")
    print("\nCommon causes:")
    print("  403 Forbidden    → SAS token wrong, expired, or missing List permission (sp=rl)")
    print("  Auth error       → Spark SAS config not set — re-run Cell 1 first")
    raise

## Cell 3 — Drill into realtime/charging_sessions folder

In [ ]:
print("Drilling into realtime/charging_sessions/\n")
try:
    sessions_root = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/realtime/charging_sessions/"
    items = dbutils.fs.ls(sessions_root)

    print("Contents of realtime/charging_sessions/:")
    for item in items:
        print(f"  {item.name}")

    # Go one level deeper
    for item in items:
        try:
            sub = dbutils.fs.ls(item.path)
            for s in sub:
                print(f"    {item.name}{s.name}")
        except:
            pass

except Exception as e:
    print(f"ERROR: {e}")
    print("→ Folder path may differ. Check Cell 2 output for correct folder names.")
    raise

## Cell 4 — Read a specific CSV file

> Adjust the date/hour in the path based on what Cell 3 shows in your container.

In [ ]:
csv_path = (
    f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
    f"/realtime/charging_sessions/2026/06/01/06/sessions_20260601_0600.csv"
)

print(f"Reading CSV:\n  {csv_path}\n")

try:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(csv_path)
    )

    print(f"Row count    : {df.count():,}")
    print(f"Column count : {len(df.columns)}")
    print(f"Columns      : {df.columns}\n")
    df.printSchema()
    display(df.limit(10))

except Exception as e:
    print(f"ERROR: {e}")
    print("\nCommon causes:")
    print("  'No such file'  → Date/hour in path doesn't match actual file. Check Cell 3 output.")
    print("  '403 Forbidden' → SAS token missing Read permission. Confirm sp=rl in token.")
    print("  'abfss error'   → Make sure you are using wasbs:// not abfss://")
    raise

## Cell 5 — Read entire charging_sessions folder (all CSVs at once)

In [ ]:
# Why glob pattern?
# The folder structure is: charging_sessions/2026/06/01/06/file.csv
# Spark cannot infer schema when it hits only subfolders at the top level.
# The wildcard pattern /*/*/*/*/*.csv tells Spark to recurse into all
# subdirectories (year/month/day/hour) and find the actual CSV files.

base      = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
glob_path = f"{base}/realtime/charging_sessions/*/*/*/*/*.csv"

print(f"Reading all CSVs using glob pattern:")
print(f"  {glob_path}\n")

try:
    df_all = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(glob_path)
    )

    total_rows = df_all.count()
    print(f"Total rows across all files : {total_rows:,}")
    print(f"Column count                : {len(df_all.columns)}")
    print(f"Columns                     : {df_all.columns}\n")
    display(df_all.limit(10))

except Exception as e:
    print(f"ERROR: {e}")
    print("\nTroubleshooting:")
    print("  1. Run Cell 3 to confirm the exact folder depth (year/month/day/hour)")
    print("  2. If depth differs, adjust the number of /* in the glob pattern")
    print("     e.g. 3 levels deep  → /*/*/*/*.csv")
    print("     e.g. 4 levels deep  → /*/*/*/*/*.csv  (current)")
    print("  3. Or read one known file first (Cell 4) to confirm access is working")
    raise

## Cell 6 — Data quality check

In [ ]:
import pyspark.sql.functions as F

print("=== Data Quality Check — Source Blob ===\n")

try:
    total = df_all.count()
    print(f"Total rows : {total:,}")
    print(f"Columns    : {len(df_all.columns)}\n")

    print("Null/empty counts per column:")
    for col_name in df_all.columns:
        null_count = df_all.filter(
            F.col(col_name).isNull() | (F.col(col_name).cast("string") == "")
        ).count()
        pct  = (null_count / total * 100) if total > 0 else 0
        flag = " ← check this" if pct > 10 else ""
        print(f"  {col_name:<35} nulls: {null_count:>6,}  ({pct:.1f}%){flag}")

    print("\nData quality check complete.")

except Exception as e:
    print(f"ERROR: {e}")
    print("Run Cell 5 first — this cell requires df_all.")

## Cell 7 — Final summary

In [ ]:
print("=" * 55)
print("SOURCE BLOB READ TEST — SUMMARY")
print("=" * 55)
print(f"  Storage account : {STORAGE_ACCOUNT}")
print(f"  Container       : {CONTAINER}")
print(f"  Protocol        : wasbs:// (SAS token auth)")
print(f"  Access          : Read + List only")
print(f"  Total rows read : {df_all.count():,}")
print(f"  Columns         : {len(df_all.columns)}")
print("=" * 55)
print("SAS token read test PASSED — source blob is accessible.")
print("\nNext: proceed to Day 2.")